In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from PIL import Image


In [8]:
# Resolve paths from the project config so the dataset location is defined in one place.
working_dir = Path.cwd().resolve()
PROJECT_ROOT = working_dir if (working_dir / 'config' / 'config.yaml').is_file() else working_dir.parent
with (PROJECT_ROOT / 'config' / 'config.yaml').open(encoding='utf-8') as config_file:
    config = yaml.safe_load(config_file)

DATA_DIR = Path(config['data_ingestion']['source_dir']).expanduser()
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Dataset directory not found: {DATA_DIR}')

IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg'}
class_dirs = [path for path in sorted(DATA_DIR.iterdir()) if path.is_dir()]
if len(class_dirs) < 2:
    raise ValueError(f'Expected at least two class folders in {DATA_DIR}')

image_table = pd.DataFrame(
    (
        {'image_path': image_path, 'class_name': class_dir.name, 'label': label}
        for label, class_dir in enumerate(class_dirs)
        for image_path in sorted(class_dir.iterdir())
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS
    )
)
if image_table.empty:
    raise ValueError(f'No supported images found in {DATA_DIR}')

class_names = tuple(class_dir.name for class_dir in class_dirs)
image_table.groupby('class_name', as_index=False).size().rename(columns={'size': 'image_count'})


,class_name,image_count
0,adenocarcinoma,195
1,normal,148


In [9]:
# Load every CT image as RGB, resize it for a CNN, and keep labels aligned with images.
IMAGE_SIZE = (224, 224)

images = np.stack([
    np.asarray(Image.open(image_path).convert('RGB').resize(IMAGE_SIZE), dtype=np.uint8)
    for image_path in image_table['image_path']
])
labels = image_table['label'].to_numpy(dtype=np.int64)

print(f'Classes (label order): {dict(enumerate(class_names))}')
print(f'Loaded images shape: {images.shape}; labels shape: {labels.shape}')
print(f'Pixel range: {images.min()} to {images.max()}')

Classes (label order): {0: 'adenocarcinoma', 1: 'normal'}
Loaded images shape: (343, 224, 224, 3); labels shape: (343,)
Pixel range: 0 to 255


In [11]:
import tensorflow as tf 

2026-08-14 18:24:52.545994: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [12]:
from pathlib import Path 
from dataclasses import dataclass 
@dataclass(frozen= True )

class PrepareBaseModelConfig : 
    root_dir : Path 
    base_model_path : Path 
    updated_base_model_path : Path 
    params_image_size : list 
    params_learning_rate: float 
    params_include_top: bool 
    params_weights: str 
    params_classes: int 
    

In [14]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories 

In [17]:
class ConfigurationManager:
    def __init__(self, config_filepath=None, params_filepath=None):
        working_dir = Path.cwd().resolve()
        project_root = working_dir if (working_dir / 'config' / 'config.yaml').is_file() else working_dir.parent
        self.project_root = project_root
        config_filepath = Path(config_filepath or project_root / 'config' / 'config.yaml')
        params_filepath = Path(params_filepath or project_root / 'params.yaml')

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        artifacts_root = Path(self.config.artifacts_root).expanduser()
        self.artifacts_root = (artifacts_root if artifacts_root.is_absolute() else project_root / artifacts_root).resolve()
        create_directories([self.artifacts_root])

    def _resolve_path(self, path_value):
        path = Path(path_value).expanduser()
        return (path if path.is_absolute() else self.project_root / path).resolve()
        
    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig : 
        config= self.config.prepare_base_model
        root_dir = self._resolve_path(config.root_dir)
        create_directories([root_dir])
        prepare_base_model_config= PrepareBaseModelConfig(
            root_dir=root_dir,
            base_model_path=self._resolve_path(config.base_model_path),
            updated_base_model_path=self._resolve_path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES 
        
        )
        return prepare_base_model_config

In [25]:
import os 
import tensorflow as tf 

In [27]:
class PrepareBaseModel:
    def __init__(self,config: PrepareBaseModelConfig):
        self.config=config 
    
    def get_base_model(self):
        self.model=tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size, 
            weights=self.config.params_weights,
            include_top= self.config.params_include_top
        )
        self.save_model(path=self.config.base_model_path,model=self.model)
        
        
        
    @staticmethod 
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

In [28]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(path=self.config.base_model_path, model=self.model)


    
    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                model.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                model.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model
    

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)
    


    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)




In [30]:
try:
    config=ConfigurationManager()
    prepare_base_model_config=config.get_prepare_base_model_config()
    prepare_base_model=PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e :
    raise e 

[2026-08-14 20:22:41]: INFO: yaml file: /Users/Shared/ThoraxGuard/config/config.yaml loaded successfully
[2026-08-14 20:22:41]: INFO: yaml file: /Users/Shared/ThoraxGuard/params.yaml loaded successfully
[2026-08-14 20:22:41]: INFO: created directory at: /Users/Shared/ThoraxGuard/artifacts
[2026-08-14 20:22:41]: INFO: created directory at: /Users/Shared/ThoraxGuard/artifacts/prepare_base_model
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [31]:
from dataclasses import dataclass 
from pathlib import Path 
@dataclass(frozen=True)
class TrainingConfig:
    root_dir :Path 
    trained_model_path:Path 
    training_data:Path 
    params_epochs:int 
    params_batch_size:int 
    params_is_augmentation:bool 
    params_image_range:list 
    

In [32]:
from cnnClassifier.constants import * 
from cnnClassifier.utils.common import read_yaml,create_directories 
import tensorflow as tf 

In [33]:
class ConfigurationManager:
    def __init__(self, config_filepath=None, params_filepath=None):
        working_dir = Path.cwd().resolve()
        project_root = working_dir if (working_dir / 'config' / 'config.yaml').is_file() else working_dir.parent
        self.project_root = project_root
        config_filepath = Path(config_filepath or project_root / 'config' / 'config.yaml')
        params_filepath = Path(params_filepath or project_root / 'params.yaml')

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        artifacts_root = Path(self.config.artifacts_root).expanduser()
        self.artifacts_root = (artifacts_root if artifacts_root.is_absolute() else project_root / artifacts_root).resolve()
        create_directories([self.artifacts_root])

    def _resolve_path(self, path_value):
        path = Path(path_value).expanduser()
        return (path if path.is_absolute() else self.project_root / path).resolve()
        
    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig : 
        config= self.config.prepare_base_model
        root_dir = self._resolve_path(config.root_dir)
        create_directories([root_dir])
        prepare_base_model_config= PrepareBaseModelConfig(
            root_dir=root_dir,
            base_model_path=self._resolve_path(config.base_model_path),
            updated_base_model_path=self._resolve_path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES 
        
        )
        return prepare_base_model_config

In [35]:
import os 
import urllib.request as request 
import time 